# PA2 matched benchmark generation
Generate Base300, SFT600, HH600 and UF600 with identical greedy penalty1.1/block0/max256 settings. Preserve the decoder-check findings and prior training versions. No training or judging in this notebook.


In [ ]:
# 1. Verify both CUDA GPUs and record the starting environment.
import json, sys, subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import torch
output = Path('/kaggle/working/pa2_outputs')
output.mkdir(exist_ok=True)
info = {'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(), 'gpu_count': torch.cuda.device_count(), 'packages': {}}
for name in ['transformers', 'datasets', 'accelerate', 'huggingface_hub', 'peft', 'trl', 'torchao']:
    try:
        info['packages'][name] = version(name)
    except PackageNotFoundError:
        info['packages'][name] = None
info['devices'] = []
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    info['devices'].append({'index': i, 'name': props.name, 'memory_gib': round(props.total_memory / 2**30, 2)})
    assert torch.ones(16, device=f'cuda:{i}').sum().item() == 16
print(json.dumps(info, indent=2))
assert info['cuda_available'] and info['gpu_count'] == 2, 'Two CUDA GPUs required'
assert all('T4' in d['name'] for d in info['devices']), 'Expected T4 devices'
# Keep the original starting environment if rerunning the setup cells.
initial = output / 'environment_before_setup.json'
if not initial.exists():
    initial.write_text(json.dumps(info, indent=2) + '\n')
print('Both T4 GPUs execute tensors.')


In [ ]:
# 2. Pin PA2 dependencies, preserve CUDA PyTorch and verify fresh Qwen imports.
torch_before = version('torch')
constraint = output / 'preserve_torch.txt'
constraint.write_text('torch==' + torch_before + '\n')
packages = ['transformers==4.57.6', 'datasets==4.8.5', 'accelerate==1.15.0', 'huggingface_hub==0.36.2',
            'tqdm', 'pyyaml', 'sentencepiece', 'safetensors', 'pytest>=8', 'matplotlib']
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                         '-c', str(constraint), *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(output / 'package_setup.log').write_text(result.stdout)
print(result.stdout[-3500:])
result.check_returncode()
assert version('torch') == torch_before, 'CUDA PyTorch must not change'
probe_command = [sys.executable, '-c', 'from transformers import AutoModelForCausalLM, AutoTokenizer; from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM; print("Qwen model imports pass")']
probe = subprocess.run(probe_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if probe.returncode and 'torchao' in probe.stdout:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    probe = subprocess.run(probe_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(probe.stdout[-3500:])
probe.check_returncode()
versions = {name: version(name) for name in ['torch', 'transformers', 'datasets', 'accelerate', 'huggingface_hub']}
(output / 'environment_after_setup.json').write_text(json.dumps(versions, indent=2) + '\n')
print(json.dumps(versions, indent=2))
print('PA2 dependencies ready. CUDA PyTorch preserved.')


In [ ]:
# Verify the saved training version and the selected checkpoints.
import hashlib, shutil, gc
def sha(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
inputs = Path('/kaggle/input')
manifests = [p for p in inputs.rglob('source_manifest.json')
             if sha(p) == '4b7deda714e222c1c0a1c10d0b0e31b85852a8828c855a8e9d3015b153d3c820']
assert len(manifests) == 1, 'Attach full DPO notebook output version 357186707'
saved_source = manifests[0].parent
root = Path('/kaggle/working/reviewed_source')
assert not root.exists(), 'Fresh session required; never overwrite source'
manifest = json.loads(manifests[0].read_text())
root.mkdir()
for name, entry in manifest['files'].items():
    file = saved_source/name
    assert len(file.read_bytes()) == entry['bytes'] and sha(file) == entry['sha256'], name
    target = root/name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(file, target)
shutil.copyfile(manifests[0], root/'source_manifest.json')
sys.path.insert(0, str(root))
from src.checkpoints import verify_checkpoint
checkpoints, identities = {}, {}
expected = {
    'sft': ('sft_full', 938, None, None),
    'hh': ('dpo_full_hh_manual_1em6', 400,
           '147b8d91e34ca3f968acaee513483fb3865abc4a522a799171076a564d7b652a',
           'd3e32f52f37dab9503d1c059104da9dea16adf5190bcb36734456d961b0dfb2c'),
    'uf': ('dpo_full_uf_manual_1em6', 469,
           'a052f64b15b866bb48def448f2af3ad215dc2eb4e30de4347281e85fabe8d3e0',
           '3945d8a7b8ed7b40308001567504c7f31b22a47e4b6a2c21daa2d8c460e6f31c')}
for track, (folder, update, run_hash, state_hash) in expected.items():
    runs = list(inputs.rglob(folder+'/run.json'))
    assert len(runs) == 1, (track, 'Ambiguous or missing checkpoint origin')
    best = runs[0].parent/'best'
    if run_hash:
        assert sha(runs[0]) == run_hash and sha(best/'state.json') == state_hash
    identity = verify_checkpoint(best)
    assert identity['state']['updates'] == update
    assert identity['state']['metadata']['track'] == track
    if track == 'sft':
        assert identity['model_sha256'] == 'bcec9f35d309873198a727c8862638b29284cf8f6bf5cef74fddd2a9d7ee026d'
    else:
        assert identity['state']['metadata']['sft_origin']['state']['files']['model.safetensors']['sha256'] == identities['sft']['model_sha256']
    checkpoints[track], identities[track] = best, identity
    print('VERIFIED CHECKPOINT', track, update, identity['model_sha256'], flush=True)
eval_root = Path('/kaggle/working/matched_evaluation')
assert not eval_root.exists(), 'Never overwrite an evaluation'
eval_root.mkdir()


In [ ]:
# Matched benchmark generation. No training; preserve earlier saved versions.
from transformers import AutoModelForCausalLM, AutoTokenizer
from scripts.generate_checkpoint import load_bank, export_checkpoint
from scripts import generate_baseline as baseline
from src.training import _write_json
from src.pairwise import build_pairs

decoding = {'do_sample': False, 'max_new_tokens': 256,
            'repetition_penalty': 1.1, 'no_repeat_ngram_size': 0}
decision = {
    'development_version': 357298097,
    'decoding': decoding,
    'decision': 'Proceed to matched measurement after user instruction to advance.',
    'limitations': [
        'No blank outputs or sustained loops in 120 reviewed development answers.',
        'UF planet-name letter-count code regresses on unique names at penalty1.1.',
        'Requested exact repetition fails; safety and reasoning failures persist.',
        'This is not an overall quality pass or a claim that DPO beats SFT.'
    ]}
_write_json(eval_root/'decoder_decision.json', decision)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cuda.matmul.allow_tf32 = False
source = {'source_manifest_sha256': sha(root/'source_manifest.json'),
          'source_hashes': {str(p.relative_to(root)): sha(p) for p in sorted((root/'src').glob('*.py'))},
          'script_sha256': sha(root/'scripts/generate_checkpoint.py'), 'seed': 42}
banks = {
    'alpaca_eval': load_bank(root/'eval_sets/alpaca_eval_300_seed42.jsonl', 'alpaca_eval'),
    'hh': load_bank(root/'eval_sets/hh_rlhf_eval_300_seed42.json', 'hh')}
_write_json(eval_root/'bank_identities.json',
            {k: {'sha256': b['sha256'], 'count': len(b['rows'])} for k, b in banks.items()})

# Regenerate the historical Base comparison under exactly the selected decoder.
# Override only this imported module's configuration; saved source stays immutable.
baseline.MAIN_DECODING = {k: v for k, v in decoding.items() if k != 'do_sample'}
common = {'revision': baseline.REVISION, 'token': False, 'trust_remote_code': False}
tokenizer = AutoTokenizer.from_pretrained(baseline.MODEL_ID, **common)
assert tokenizer.eos_token_id == 151643
model = AutoModelForCausalLM.from_pretrained(
    baseline.MODEL_ID, torch_dtype=torch.float32, **common).to('cuda:0')
baseline.export(model, tokenizer, banks['alpaca_eval']['rows'], eval_root/'base-alpaca_eval')
_write_json(eval_root/'base-alpaca_eval'/'source.json', source)
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

for track in ['sft', 'hh', 'uf']:
    model = AutoModelForCausalLM.from_pretrained(checkpoints[track], local_files_only=True,
                trust_remote_code=False, torch_dtype=torch.float32).to('cuda:0')
    tokenizer = AutoTokenizer.from_pretrained(checkpoints[track], local_files_only=True,
                                             trust_remote_code=False)
    assert verify_checkpoint(checkpoints[track]) == identities[track], 'Checkpoint changed'
    for bank_name, bank in banks.items():
        print('START FIXED BANK', track, bank_name, flush=True)
        result = export_checkpoint(model, tokenizer, bank, track, identities[track],
                     eval_root/(track+'-'+bank_name), decoding, source)
        assert result['status'] == 'completed' and result['completed'] == 300
        print('COMPLETED FIXED BANK', track, bank_name, flush=True)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# Build all four blind comparisons only after matched generation completes.
# build_pairs validates identity, IDs, complete instructions, decoding and hashes.
for track in ['hh', 'uf']:
    for bank_name in banks:
        pairs, mapping = build_pairs(eval_root/('sft-'+bank_name),
                         eval_root/(track+'-'+bank_name),
                         eval_root/('pairs-'+track+'-'+bank_name), seed=42)
        assert len(pairs) == 300
        print('SAVED BLIND PAIRS', track, bank_name, len(pairs), flush=True)
_write_json(eval_root/'completed.json', {'status': 'completed', 'responses': 2100,
                                       'blind_pairs': 1200, 'decoding': decoding})
print('MATCHED BENCHMARK GENERATION COMPLETE. Blind judging is a separate run.', flush=True)
